# Qwen3-8B Baseline 与 TT 模型评测

使用同一个模型实例和同一个 `EvaluationConfig`，依次评测 Baseline 模型与按 `LAYER_INDICES` 批量替换 `down_proj` 后的 TT 模型。默认评测第 0～5 层，并读取 Wikitext 正式全量配置；切换 `EVALUATION_CONFIG_PATH` 可改用 HellaSwag，快速试跑可在 `from_json` 中覆盖 `limit=1`。

本 Notebook 不保存 JSON。修改 `src` 后请从头 **Run All**，不要只运行后半部分 cell，以免混用修改前后的类和实验状态。

In [ ]:
import gc
import sys
from pathlib import Path
from pprint import pprint

import torch

PROJECT_ROOT = Path("/home/xls/workspace/projects/qwen3-tn-compression").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ipython = get_ipython()
ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

old_experiment = globals().get("experiment")
if old_experiment is not None:
    old_experiment.close()
for variable_name in ("experiment", "model", "tokenizer"):
    globals().pop(variable_name, None)
del old_experiment
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from qwen3_tn import TTMatrixExperiment, build_qwen_mlp_tt_matrix_targets
from qwen3_tn.model_evaluation import EvaluationConfig, evaluate_model

MODEL_PATH = Path("/infini-data/Qwen3-8B")
LAYER_INDICES = list(range(0, 6))  # 全部 36 层可改为 list(range(36))
PROJECTION_CONFIGS = {
    "down_proj": {
        "out_modes": (8, 8, 8, 8),
        "in_modes": (8, 8, 8, 24),
        "ranks": (1, 64, 2048, 192, 1),
        "token_chunk_size": 8,
    }
}
LAYER_OVERRIDES = {
    # (1, "down_proj"): {"ranks": (1, 64, 1024, 192, 1)},
}
SVD_DRIVER = "gesvd"
MIN_FREE_GPU_GIB = 20
CACHE_ROOT = PROJECT_ROOT / "artifacts/tt_matrix_cache"
FORCE_RECOMPUTE = False
EVALUATION_CONFIG_PATH = PROJECT_ROOT / "configs/evaluation/wikitext.json"
config = EvaluationConfig.from_json(
    EVALUATION_CONFIG_PATH,
    # limit=1,  # 快速试跑时取消注释；正式评测使用文件中的 null。
)
TASK_DISPLAY_METRICS = {
    "wikitext": (
        "word_perplexity,none",
        "bits_per_byte,none",
    ),
    "hellaswag": (
        "acc,none",
        "acc_norm,none",
    ),
}

if not torch.cuda.is_available():
    raise RuntimeError("该评测需要 CUDA GPU")
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"模型目录不存在：{MODEL_PATH}")

targets = build_qwen_mlp_tt_matrix_targets(
    LAYER_INDICES,
    PROJECTION_CONFIGS,
    LAYER_OVERRIDES,
)
experiment = TTMatrixExperiment(
    MODEL_PATH,
    device="cuda:0",
    model_dtype=torch.bfloat16,
    core_dtype=torch.bfloat16,
    min_free_gpu_gib=MIN_FREE_GPU_GIB,
)
model = experiment.load_model()
tokenizer = experiment.tokenizer
assert tokenizer is not None

print("项目目录：", PROJECT_ROOT)
print("模型：", MODEL_PATH)
print("目标数量：", len(targets))
for target in targets:
    print(target.module_path, target.ranks)
print("评测配置：")
pprint(config)

/home/xls/appdata/miniforge3/envs/qwen3-tn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 5/5 [00:05<00:00,  1.14s/it]

项目目录： /mnt/intern7/xls/projects/qwen3-tn-compression
模型： /infini-data/Qwen3-8B
目标数量： 6
model.layers.0.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.1.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.2.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.3.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.4.mlp.down_proj (1, 64, 2048, 192, 1)
model.layers.5.mlp.down_proj (1, 64, 2048, 192, 1)
评测配置：
EvaluationConfig(task='wikitext',
                 limit=None,
                 batch_size=1,
                 max_length=2048,
                 num_fewshot=0,
                 apply_chat_template=False,
                 bootstrap_iters=0,
                 seed=42)


In [2]:
experiment.restore()
baseline_result = evaluate_model(
    model,
    config,
    tokenizer=tokenizer,
)

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


开始评测任务：wikitext


[Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
[Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
[Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
[Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
[Task: wikitext] metric bits_per_byte is defined, but aggregation is not. using default aggregation=bits_per_byte
[Task: wikitext] metric bits_per_byte is defined, but higher_is_better is not. using default higher_is_better=False
Overwriting default num_fewshot of wikitext from None to 0
Running loglikelihood requests: 100%|██████████| 1/1 [00:00<00:00,  5.13it/s]
fatal: not a git repository (or any parent up to mount point /mnt)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


任务评测完成：wikitext，耗时 52.26 秒


In [3]:
decomposition = experiment.decompose(
    targets,
    svd_driver=SVD_DRIVER,
    cache_root=CACHE_ROOT,
    force_recompute=FORCE_RECOMPUTE,
    verbose=True,
)
print("TT 分解汇总：")
pprint(decomposition["aggregate"])

experiment.install()
try:
    tt_result = evaluate_model(
        model,
        config,
        tokenizer=tokenizer,
    )
finally:
    restored = experiment.restore()
    print("TT 评测结束，已恢复 Baseline 模型层：", restored)

[1/6] model.layers.0.mlp.down_proj：检查缓存元数据
[1/6] model.layers.0.mlp.down_proj：元数据命中，验证权重 SHA-256
[1/6] model.layers.0.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[2/6] model.layers.1.mlp.down_proj：检查缓存元数据
[2/6] model.layers.1.mlp.down_proj：元数据命中，验证权重 SHA-256
[2/6] model.layers.1.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[3/6] model.layers.2.mlp.down_proj：检查缓存元数据
[3/6] model.layers.2.mlp.down_proj：元数据命中，验证权重 SHA-256
[3/6] model.layers.2.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[4/6] model.layers.3.mlp.down_proj：检查缓存元数据
[4/6] model.layers.3.mlp.down_proj：元数据命中，验证权重 SHA-256
[4/6] model.layers.3.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[5/6] model.layers.4.mlp.down_proj：检查缓存元数据
[5/6] model.layers.4.mlp.down_proj：元数据命中，验证权重 SHA-256


[5/6] model.layers.4.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[6/6] model.layers.5.mlp.down_proj：检查缓存元数据
[6/6] model.layers.5.mlp.down_proj：元数据命中，验证权重 SHA-256
[6/6] model.layers.5.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
TT 分解汇总：
{'cache_hit_count': 6,
 'cache_load_seconds': 0.003815529402345419,
 'compressed_model_parameters': 8090317824,
 'decomposed_count': 0,
 'dense_target_parameters': 301989888,
 'matrix_error_frobenius_norm_squared': 35802.91542947695,
 'matrix_reference_frobenius_norm_squared': 138687.19733680828,
 'matrix_weight_all_finite': True,
 'matrix_weight_relative_l2': 0.5080904247125808,
 'model_compression_ratio': 1.012412063182748,
 'model_parameter_reduction': 100417536,
 'model_parameter_reduction_fraction': 0.012259892620923307,
 'original_model_parameters': 8190735360,
 'target_compression_ratio': 1.4981711777615216,
 'target_count': 6,
 'target_parameter_reduction': 100417536,
 'target_parameter_reduction_fraction': 0.33251953125,
 'tt_matrix_target_parameter

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


开始评测任务：wikitext


[Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
[Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
[Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
[Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
[Task: wikitext] metric bits_per_byte is defined, but aggregation is not. using default aggregation=bits_per_byte
[Task: wikitext] metric bits_per_byte is defined, but higher_is_better is not. using default higher_is_better=False
Overwriting default num_fewshot of wikitext from None to 0
Running loglikelihood requests: 100%|██████████| 1/1 [00:03<00:00,  3.70s/it]
fatal: not a git repository (or any parent up to mount point /mnt)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


任务评测完成：wikitext，耗时 617.60 秒
TT 评测结束，已恢复 Baseline 模型层： True


In [4]:
try:
    baseline_metrics = baseline_result["raw_results"]["results"][config.task]
    tt_metrics = tt_result["raw_results"]["results"][config.task]

    metric_names = TASK_DISPLAY_METRICS.get(config.task)
    if metric_names is None:
        raise KeyError(f"请先在 TASK_DISPLAY_METRICS 中配置任务：{config.task}")
    missing_metric_names = [
        name
        for name in metric_names
        if name not in baseline_metrics or name not in tt_metrics
    ]
    if missing_metric_names:
        raise KeyError(f"评测结果缺少展示指标：{missing_metric_names}")

    metric_rows = [
        (model_name, metric_name, metrics[metric_name])
        for model_name, metrics in (
            ("Baseline", baseline_metrics),
            ("TT", tt_metrics),
        )
        for metric_name in metric_names
    ]

    print(f"{'模型':<10} {'指标':<30} {'值':>12}")
    print("-" * 56)
    for model_name, metric_name, metric_value in metric_rows:
        print(f"{model_name:<10} {metric_name:<30} {metric_value:>12.6f}")
finally:
    experiment.close()
    globals().pop("model", None)
    globals().pop("tokenizer", None)
    gc.collect()
    torch.cuda.empty_cache()

模型         指标                                        值
--------------------------------------------------------
Baseline   word_perplexity,none              13.488154
Baseline   bits_per_byte,none                 0.701946
TT         word_perplexity,none              14.021326
TT         bits_per_byte,none                 0.712405
